In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
from delta.tables import DeltaTable

# First, write the CSV data as a Delta table
df.write.format("delta").mode("overwrite").save("/Volumes/external-catalog/default/test-volume/Employee_Attrition_delta")

# Now read the Delta table history
delta_table = DeltaTable.forPath(spark, "/Volumes/external-catalog/default/test-volume/Employee_Attrition_delta")
history_df = delta_table.history()
display(history_df.select("version"))

In [0]:
df_delta = spark.read.format("delta").load("/Volumes/external-catalog/default/test-volume/Employee_Attrition_delta")
display(df_delta)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data
COMMENT 'Managed volume for transformed employee data'
""")

In [0]:
from pyspark.sql.functions import col, when

# Logical transformation: Add a new column 'IsHighIncome' based on MonthlyIncome
df_transformed = df_delta.withColumn(
    "IsHighIncome",
    when(col("MonthlyIncome") > 10000, "Yes").otherwise("No")
)

# Write the transformed DataFrame into the volume partitioned by Department
df_transformed.write.partitionBy("Department").format("delta").mode("overwrite").save(
    "/Volumes/external-catalog/default/employee_transformed_data"
)